In [4]:
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
# from langgraph.prelude import Send
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI  # 或你用的任何模型
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnablePassthrough
from langchain.chat_models import init_chat_model


from ragPipeline.rag_pipeline import rag_Pipeline
import operator
from dotenv import load_dotenv
import pandas as pd
from geopy.distance import geodesic
from src.model import Query
import json
import os

load_dotenv()

True

In [3]:
# =====================================
# 1. 定义状态（State） - 最核心的部分
# =====================================
class DriverReportState(TypedDict):
    # 输入数据（通常从外部传入）
    driver_id: str
    start_date: str                # e.g. "2026-03-09"
    end_date: str                  # e.g. "2026-03-15"
    raw_data: pd.DataFrame           # 原始 telematics 数据（GPS、速度、怠速、急刹等）
    llm: BaseChatModel
    rag: rag_Pipeline

    # 中间结果
    metrics_summary: dict          # 统计指标：总里程、平均速度、超速次数、怠速时长...
    analysis_result = dict          # result from detectAnomalies
    anomalies: list[dict]          # 异常事件列表 [{type, time, severity, description}, ...]
    anomaly_level: str             # "low", "medium", "high", "critical"
    report_draft: str              # LLM 初稿
    final_report: str              # 最终版本（可能经过人工修改）
    messages: Annotated[Sequence[BaseMessage], operator.add]  # 用于调试/追踪的对话历史

    # 控制流相关
    needs_human_review: bool
    human_feedback: str | None

In [5]:
# =====================================
# 2. 定义 Tools（可选，示例）
# =====================================
def read_file() -> pd.DataFrame:
    """读取文件内容"""
    df = pd.read_csv("src/Telematicsdata.csv")
    
    df['timeMili'] = pd.to_datetime(df['timeMili'], unit='ms')
    
    # print(df)
    
    return df
  
# read_file()

# calculate distance by gps coordinates    
def calculate_distance_by_row(row):
    # 获取当前点和上一个点的坐标
    lat1, lon1 = row['prev_lat'], row['prev_lon']
    lat2, lon2 = row['latitude'], row['longitude']
    
    # 如果是第一行，没有前一个点，距离为 0
    if pd.isna(lat1):
        return 0.0
    
    # 使用 geodesic 计算米数 (也可以使用 .miles 计算英里)
    return geodesic((lat1, lon1), (lat2, lon2)).meters

# @tool
def query_telematics_data(driver_id: str, start_date: str, end_date: str) -> pd.DataFrame:
    """从数据库或数据湖查询该司机一周的 telematics 原始数据"""
    # 实际实现：调用 SQL / BigQuery / API
    df = read_file()
    
    # 关键：转成 datetime 对象再比较
    start = pd.to_datetime(start_date)
    end   = pd.to_datetime(end_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)  # 包含 end 当天最后一秒
    
    df = df[(df['deviceId'] == driver_id) & (df['timeMili'] >= start) & (df['timeMili'] <= end)]

    return df



def calculate_distance(df: pd.DataFrame) -> float:
    """计算总里程、平均速度等统计指标"""
    
    df = df[df['variable'] == 'POSITION']
    
    df['latitude'] = df['value'].str.split(',').str[0].astype(float)
    df['longitude'] = df['value'].str.split(',').str[1].astype(float)
    
    df['prev_lat'] = df['latitude'].shift(1)
    df['prev_lon'] = df['longitude'].shift(1)
    
    # speed could be slow if the dataset is large, consider using a more efficient method or parallel processing
    df['distance_m'] = df.apply(calculate_distance_by_row, axis=1)
    
    total_distance = df['distance_m'].sum()
    
    print(df[['timeMili', 'latitude', 'longitude', 'distance_m']])
    
    # 其他指标...
    return total_distance

print(calculate_distance(query_telematics_data('zRYzhAEAHAABAAAKCRtcAAsAtwB1gBAQ','2020-08-16','2020-08-17')))

def calculate_average_speed(df: pd.DataFrame) -> float:
    """计算平均速度"""
    df = df[df['variable'] == 'Vehicle speed']
    df['speed'] = df['value'].astype(float)
    average_speed = df['speed'].mean()
    return average_speed

print(calculate_average_speed(query_telematics_data('zRYzhAEAHAABAAAKCRtcAAsAtwB1gBAQ','2020-08-16','2020-08-17')))

def count_over_speed(df: pd.DataFrame) -> int:
    """计算超速次数"""
    df = df[df['variable'] == 'Vehicle speed']
    df['speed'] = df['value'].astype(float)
    over_speed_trips = df[df['speed'] > 80].shape[0]
    return over_speed_trips

print(count_over_speed(query_telematics_data('zRYzhAEAHAABAAAKCRtcAAsAtwB1gBAQ','2020-08-16','2020-08-17')))

def count_idling_time(df: pd.DataFrame) -> int:
    """计算怠速时长"""
    df = df[(df['variable'] == 'IDLING') & (df['value'] == 1)]
    
    idling_time = df.shape[0]
    return idling_time
print(count_idling_time(query_telematics_data('zRYzhAEAHAABAAAKCRtcAAsAtwB1gBAQ','2020-08-16','2020-08-17')))

# @tool
# def send_email_notification(driver_id: str, report_summary: str, severity: str):
#     """当 critical 异常时发送邮件给主管"""
#     print(f"[模拟] 邮件已发送给主管：司机 {driver_id} 周报 - {severity} 级别异常")
#     return "Notification sent"



                 timeMili   latitude  longitude  distance_m
1     2020-08-16 18:30:03  13.330059  74.744670         0.0
2     2020-08-16 18:30:07  13.330059  74.744670         0.0
3     2020-08-16 18:30:10  13.330059  74.744670         0.0
5     2020-08-16 18:30:14  13.330059  74.744670         0.0
6     2020-08-16 18:30:17  13.330059  74.744670         0.0
...                   ...        ...        ...         ...
48306 2020-08-17 18:29:43  13.330205  74.744026         0.0
48310 2020-08-17 18:29:46  13.330205  74.744026         0.0
48311 2020-08-17 18:29:50  13.330205  74.744026         0.0
48313 2020-08-17 18:29:53  13.330205  74.744026         0.0
48314 2020-08-17 18:29:56  13.330205  74.744026         0.0

[22357 rows x 4 columns]
50252348.788309015
21.981225296442688
0
0


In [6]:
def fetechAndSummary(state: DriverReportState) -> DriverReportState:
    """示例函数：获取数据并生成统计摘要"""
    # 1. 获取原始数据
    df = query_telematics_data(state['driver_id'], state['start_date'], state['end_date'])
    
    
    
    # 2. 生成统计摘要（这里是 mock，实际会更复杂）
    state['metrics_summary'] = {
        "total_distance": float(calculate_distance(df)),  # m
        "average_speed": float(calculate_average_speed(df)),     # km/h
        "overspeed_events": count_over_speed(df),
        "idle_time": count_idling_time(df)
    }
    
    # print(state['metrics_summary'])
    
    return state

def detectAnomalies(state: DriverReportState) -> DriverReportState:
    
    context = state['rag'].rerank("rule for vehicle speed and distance",state['rag'].retrieve("rule for vehicle speed and distance",5),3)
     
    
    prompt = ChatPromptTemplate.from_messages([
        ('system',"""
         You are a professional driver safety expert. You are given a list of events and a context. Your task is to detect any anomaly in the events.
         If there is an anomaly, you should return a list of anomalies.
         
         Must be follwing these rules otherwise task will fail: 
                1. Output only pure JSON objects; no prefixes, suffixes, markdown, or ```json` are allowed.
                2. All double quotes in the string must be escaped as `\"` (backslash followed by a double quote). Single quotes do not need to be escaped. Backslashes `**` must be escaped as `\\**`.
                3. Content in the `event_value` field can have line breaks, but the line break character must be `\n`.
                4. Unescaped double quotes, line breaks, control characters, and other content that could disrupt the JSON structure are prohibited.
                5. JSON strings must be completely closed; commas and parentheses are not allowed.
                
            {{
                "overview": 'Good' | 'Bad' | 'moderate'
                "event": [
                    {{
                        "event_type": "overspeed",
                        "event_value": "2023-01-01 00:00:00",
                    }}
                ]
            }}
         
         """),
        ('human',""" Driver data: {input_data}, analyze any anomaly based on context: {input_context}.
         """)
    ])
    
    chain = (
        RunnablePassthrough.assign(
            input_data = lambda _: json.dumps(state['metrics_summary'], ensure_ascii=False, indent=2),
            input_context = lambda x: x["input_context"]
        )
        | prompt
        | state['llm'].with_structured_output(Query)
    )
    
    result = chain.invoke({"input_context": context})
    print(result)
    
    return {"analysis_result":result.model_dump()}

In [7]:
app = StateGraph(DriverReportState)

app.add_node("fetch_data",fetechAndSummary)
app.add_node("detect_anomalies",detectAnomalies)

app.add_edge(START, "fetch_data")
app.add_edge("fetch_data", "detect_anomalies")
app.add_edge("detect_anomalies", END)

checkPoint = MemorySaver()
agent = app.compile(checkpointer=checkPoint)

llm = init_chat_model(
        "gemini-2.5-flash",               # 直接写模型名
        model_provider="google_genai",
        google_api_key=os.getenv("GEMINI_API_KEY"),
        temperature=0.2,
        max_output_tokens=8192,
        # 其他参数如 thinking_level="low"（新功能，视版本支持）
    )
rag = rag_Pipeline()

initial_state = {'driver_id':'zRYzhAEAHAABAAAKCRtcAAsAtwB1gBAQ','start_date':'2020-08-16','end_date':'2020-08-17','llm':llm,'rag':rag}

thread = {"configurable": {"thread_id": "daily_report_zRYzhAEAHAABAAAKCRtcAAsAtwB1gBAQ"}}

for event in agent.stream(initial_state, thread, stream_mode="values"):
    print(event)




Loading weights: 100%|██████████| 310/310 [00:00<00:00, 18235.34it/s]


Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.